In [2]:
import torch
import torch.nn as nn
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data pre-processing

In [3]:
transform=transforms.ToTensor()
train_dataset=datasets.MNIST(root="data",train=True,download=True,transform=transform)
test_dataset=datasets.MNIST(root="data",train=False,download=True,transform=transform)
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=64)
test_loader=DataLoader(test_dataset,shuffle=False,batch_size=64)
print(f"Shape of data: {next(iter(train_loader))[0].shape}")

Shape of data: torch.Size([64, 1, 28, 28])


## Model Block

In [4]:
class Resnet_Block(nn.Module):
    def __init__(self,channel):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=channel,out_channels=channel,kernel_size=3,stride=1,padding=1)
        self.conv2=nn.Conv2d(in_channels=channel,out_channels=channel,kernel_size=3,stride=1,padding=1)
        self.conv3=nn.Conv2d(in_channels=channel,out_channels=channel,kernel_size=3,stride=1,padding=1)
        self.conv4=nn.Conv2d(in_channels=channel,out_channels=channel,kernel_size=3,stride=1,padding=1)
        self.n1=nn.BatchNorm2d(channel)
        self.n2=nn.BatchNorm2d(channel)
        self.n3=nn.BatchNorm2d(channel)
        self.n4=nn.BatchNorm2d(channel)
        self.relu=nn.ReLU()
    def forward(self,x):
        identity=x
        x=self.conv1(x)
        x=self.n1(x)
        x=self.relu(x)

        x=self.conv2(x)
        x=self.n2(x)
        x=self.relu(x)

        x=self.conv3(x)
        x=self.n3(x)
        x=self.relu(x)

        x=self.conv4(x)
        x=self.n4(x)
        x=identity+x
        x=self.relu(x)
        return x


## Model

In [5]:
class Resnet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,16,kernel_size=3,padding=1,stride=1)
        self.conv2=nn.Conv2d(16,16,kernel_size=16,padding=1,stride=1)
        self.n1=nn.BatchNorm2d(16)
        self.n2=nn.BatchNorm2d(16)
        self.relu=nn.ReLU()
        self.block1=Resnet_Block(16)
        self.block2=Resnet_Block(16)
        self.fc=nn.Linear(16,10)
        self.pool=nn.AdaptiveAvgPool2d((1, 1))
    def forward(self,x):
        x=self.conv1(x)
        x=self.block1(x)
        x=self.n1(x)
        x=self.relu(x)

        x=self.conv2(x)
        x=self.block2(x)
        x=self.n2(x)
        x=self.relu(x)

        x=self.pool(x)

        x=torch.flatten(x,1)
        x=self.fc(x)
        return x

## Training

In [12]:
epoch=3
model=Resnet()
model=model.to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=0.0001)
print("Training loss")
model.train()
for i in range(epoch):
    loss=0
    total=0
    for data,result in train_loader:
        data=data.to(device)
        result=result.to(device)
        optimizer.zero_grad()
        predicted_result=model(data)
        batch_loss=loss_fn(predicted_result,result)
        batch_loss.backward()
        optimizer.step()
        loss+=batch_loss.item()
        total+=result.size(0)
    print(f"{i} : {loss/total}")
model.eval()
loss=0
total=0
for data,result in test_loader:
    data=data.to(device)
    result=result.to(device)
    predicted_result=model(data)
    actual_result=torch.argmax(predicted_result,dim=1)
    loss+=(actual_result==result).sum().item()
    total+=result.size(0)
print(f"Test accuracy: {loss/total}")

Training loss
0 : 0.01846320330798626
1 : 0.007883218991259733
2 : 0.0040318756210307284
Test accuracy: 0.9881
